## Method 1
#### Agent help to suggest the filename with the help of prompt and also the link.

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_groq.chat_models import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",    
    # reasoning_effort= "parsed"  # disable thinking
    reasoning_format="hidden",
)

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25682.97it/s]


In [3]:
from langchain_community.document_loaders import JSONLoader

json_data = JSONLoader(
    "../web-crawller/links_result_deduped.json",
    jq_schema=".[]",
    text_content=True
).load()
json_data

[Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 1}, page_content='https://ardupilot.org/copter/docs'),
 Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 2}, page_content='https://ardupilot.org/copter/docs/ArduCopter_MAVLink_Messages.html'),
 Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 3}, page_content='https://ardupilot.org/copter/docs/ac2_followme.html'),
 Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 4}, page_content='https://ardupilot.org/copter/docs/ac2_guidedmode.html'),
 Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 5}, page_content='https://ardupilot.org/copter/docs/ac2_positionmode.html'),
 Document(metadata={'source

In [6]:
json_data[930]

Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 931}, page_content='https://ardupilot.org/copter/docs/www.DJI.com')

In [12]:
from langchain_community.vectorstores import Chroma

links_vector_db = Chroma.from_documents(
    documents=json_data,
    embedding=embedding
)

In [17]:
link_retriever = links_vector_db.as_retriever()
link_retriever.invoke("parameters")

[Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 827}, page_content='https://ardupilot.org/copter/docs/parameters.html'),
 Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 569}, page_content='https://ardupilot.org/copter/docs/common-scripting-parameters.html'),
 Document(metadata={'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 897}, page_content='https://ardupilot.org/copter/docs/traditional-helicopter-parameter-list.html'),
 Document(metadata={'seq_num': 774, 'source': '/home/administrator/personal/arduchat/web-crawller/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/parameters-Copter-beta-V4.6.3.html')]

In [29]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a senior drone engineer and you are mentoring the juniors on the Ardupilot Copter documentation. 

Here is the junior question : {question},

now your task is to give set of keywords so that retriever can easily find the links that may be necessary for the answering the questions
Just give a array, donot give any thing else
""")

prompt.invoke({"question": "what is circle mode"})

StringPromptValue(text='\nYou are a senior drone engineer and you are mentoring the juniors on the Ardupilot Copter documentation. \n\nHere is the junior question : what is circle mode,\n\nnow your task is to give set of keywords so that retriever can easily find the links that may be necessary for the answering the questions\nJust give a array, donot give any thing else\n')

In [30]:
prompt.invoke({"question":"what is circle mode"})

StringPromptValue(text='\nYou are a senior drone engineer and you are mentoring the juniors on the Ardupilot Copter documentation. \n\nHere is the junior question : what is circle mode,\n\nnow your task is to give set of keywords so that retriever can easily find the links that may be necessary for the answering the questions\nJust give a array, donot give any thing else\n')

In [31]:
chain_keyword = prompt | llm

In [68]:
link_keyword = chain_keyword.invoke({"question":"what is circle mode"}).content

In [69]:
link_docs = link_retriever.invoke(link_keyword)
retrieved_urls = [doc.page_content for doc in link_docs]
retrieved_urls

['https://ardupilot.org/copter/docs/flight-modes.html',
 'https://ardupilot.org/copter/docs/mission-command-list.html',
 'https://ardupilot.org/copter/docs/terrain-following-manual-modes.html',
 'https://ardupilot.org/copter/docs/flight-options.html']

In [66]:
already_fetched_urls = set()

url_to_be_searched = []

def add_retrieved_urls(new_urls):
    global url_to_be_searched
    url_to_be_searched = [u for u in new_urls if u not in already_fetched_urls]

def mark_as_fetched():
    already_fetched_urls.update(url_to_be_searched)

In [ ]:
add_retrieved_urls(retrieved_urls)
mark_as_fetched()

['https://ardupilot.org/copter/docs/mission-command-list.html', 'https://ardupilot.org/copter/docs/terrain-following-manual-modes.html', 'https://ardupilot.org/copter/docs/flight-options.html']
{'https://ardupilot.org/copter/docs/mission-command-list.html', 'https://ardupilot.org/copter/docs/flight-options.html', 'https://ardupilot.org/copter/docs/terrain-following-manual-modes.html', 'https://ardupilot.org/copter/docs/common-wiki_editing_guide.html', 'https://ardupilot.org/copter/docs/pdf-guides.html', 'https://ardupilot.org/copter/docs/circle-mode.html', 'https://ardupilot.org/copter/docs/flight-modes.html'}


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

docs = WebBaseLoader(url_to_be_searched).load()
docs

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://ardupilot.org/copter/docs/mission-command-list.html', 'title': 'Copter Mission Command List — Copter  documentation', 'language': 'en'}, page_content='\n\n\n\n\nCopter Mission Command List — Copter  documentation\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nHome\n\nCopter\nPlane\nRover\nBlimp\nSub\nAntennaTracker\nMission Planner\nAPM Planner 2\nMAVProxy\nCompanion Computers\nDeveloper\n\n\nDownloads\n\nMission Planner\nAPM Planner 2\nAdvanced User Tools\nDeveloper Tools\nFirmware\n\n\nCommunity\n\nSupport Forums\nFacebook\nDeveloper Chat (Discord)\nDeveloper Voice (Discord)\nContact us\nGetting involved\nCommercial Support\nDevelopment Team\nUAS Training Centers\n\n\nStores\nAbout\n\nNews\nHistory\nLicense\nTrademark\nAcknowledgments\nWiki Editing Guide\nPartners Program\n\n\n\n\n\n\n\n\n\n            Copter\n              \n\n\n\n\n\n\n\n\n\n\nIntroduction to Copter\nChoosing an Autopilot\nGround Control Stations\nChoosing a Ground Station\nMiss

In [75]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splitted_docs = splitter.split_documents(docs)

In [76]:
vectordb = Chroma.from_documents(
    documents = splitted_docs,
    embedding = embedding
)
vectordb

In [ ]:
main_retriever = vectordb.as_retriever()

In [81]:
prompt2 = PromptTemplate.from_template(
    """Help me answer this question :{question} with the help of context:{context}
    """
)

In [82]:
from langchain_core.runnables import RunnablePassthrough
chain = (
    {"context": main_retriever, "question": RunnablePassthrough()}
    | prompt2
    | llm
)

In [84]:
result = chain.invoke("what is circle mode")
from IPython.display import Markdown
Markdown(result.content)

**Circle Mode** in ArduPilot (for drones like Copter) is a flight mode where the drone circles around a specified location a set number of times before proceeding to the next mission command. Here's a detailed breakdown based on the provided context:

1. **Core Functionality**:  
   - The drone **loiters (or circles)** around a designated point, completing a specified number of full rotations.  
   - After completing the required turns, it continues to the next command in its mission.

2. **Key Parameters**:  
   - **Radius**:  
     - Determines the distance from the target point (in meters).  
     - A radius of **0** means the drone hovers in place and performs pirouettes (spins) without moving.  
     - **Negative values** reverse the direction of rotation (e.g., counter-clockwise instead of clockwise).  
     - Radii over **255 meters** are rounded down to the nearest 10 meters (e.g., 300m → 290m).  

   - **Number of Turns**:  
     - Fractional turns **between 0 and 1** (e.g., 0.5 turns) are allowed for partial rotations.  
     - For **1 or more full turns**, only whole numbers are valid.  

   - **Location**:  
     - If latitude/longitude is set to **0**, the drone uses its current position as the center of the circle.  
   - **Altitude**:  
     - If altitude is **0**, the drone maintains its current altitude while circling.  

3. **Mission Integration**:  
   - Circle Mode is part of the **mission command list**, enabling drones to autonomously circle a location during pre-programmed missions.  
   - The drone transitions to the next mission command once it intersects the path to the next target while circling.  

4. **Use Cases**:  
   - Surveillance (e.g., inspecting a structure by circling it).  
   - Delaying movement while maintaining position (e.g., waiting for a signal).  
   - Testing autonomous navigation or sensor behavior during circular motion.  

**Limitations**:  
- Radius values over 255 meters are truncated for technical reasons.  
- Direction is controlled via the sign of the radius parameter, not a separate input.  

This mode is distinct from Loiter Mode, which focuses on maintaining position without structured circular motion. Circle Mode provides precise, repeatable rotations for specific mission requirements.